### Step 1: Import Libraries
Load required packages for file handling, data processing, and logging.

In [1]:
from pathlib import Path
import pandas as pd
import logging
import re
import unicodedata
import sys
import glob
import numpy as np
import os

### Step 2: Define Project Paths
Set the root directory and paths for the dataset and pipeline folders.

In [2]:
# Project root
PROJECT_ROOT = Path(r"C:\Users\jalleyne\Desktop\ProteXXa\data_cleaning_scripts")

# Main dataset folder
DATASET_DIR = PROJECT_ROOT / "New Meet_dataset" / "new.meet.dataset"
LOGS_DIR = PROJECT_ROOT / "New Meet_dataset" / "new.meet.dataset_logs"

# Subfolders
RAW_DIR = DATASET_DIR / "new_meet_raw"
PROCESSED_DIR = DATASET_DIR / "new_meet_processed"
GARBAGE_DIR = DATASET_DIR / "new_meet_garbage"
INGESTED_DIR = DATASET_DIR / "new_meet_ingested"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("RAW_DIR:", RAW_DIR)

PROJECT_ROOT: C:\Users\jalleyne\Desktop\ProteXXa\data_cleaning_scripts
DATASET_DIR: C:\Users\jalleyne\Desktop\ProteXXa\data_cleaning_scripts\New Meet_dataset\new.meet.dataset
RAW_DIR: C:\Users\jalleyne\Desktop\ProteXXa\data_cleaning_scripts\New Meet_dataset\new.meet.dataset\new_meet_raw


### Step 3: Create Folder Structure
Ensure all required directories exist for raw, processed, garbage, and ingested data.

In [3]:

ROOT = Path.cwd()

# ===============================
# Logs setup (MUST come first)
# ===============================
LOGS_DIR = ROOT / "NEW MEET_logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOGS_DIR / "setup.log"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Starting folder setup process.")

# ===============================
# Dataset base folder
# ===============================
DATASET_DIR = ROOT / "NEW MEET_dataset"
NEW_MEET_RAW_DIR = DATASET_DIR / "new meet_raw"

# ===============================
# Subfolders
# ===============================
RAW_DIR = NEW_MEET_RAW_DIR / "new meet_raw"
PROCESSED_DIR = NEW_MEET_RAW_DIR / "new meet_processed"
GARBAGE_DIR =NEW_MEET_RAW_DIR / "new meet_garbage"
INGESTED_DIR = NEW_MEET_RAW_DIR / "new meet_ingested"

# ===============================
# Folder creation function
# ===============================
def create_folders():
    folders = [
        RAW_DIR,
        PROCESSED_DIR,
        GARBAGE_DIR,
        INGESTED_DIR,
        LOGS_DIR,
    ]

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)
        logging.info(f"Ensured folder exists: {folder}")

# ===============================
# Create the folders
# ===============================
create_folders()

### Step 4: Verify Raw Data Folder
Confirm the raw data file is present in the raw directory.

In [4]:
print("Files inside RAW_DIR:")
print(os.listdir(RAW_DIR))

Files inside RAW_DIR:
['new_meet_raw.csv']


# Load the raw CSV for inspection

In [5]:
CSV_FILE = RAW_DIR / "new_meet_raw.csv"

if CSV_FILE.exists():
    print(f"✓ CSV file found: {CSV_FILE}")
    print(f"File size: {CSV_FILE.stat().st_size:,} bytes")
else:
    raise FileNotFoundError(f"Missing file: {CSV_FILE}")

# Read using semicolon separator
df_raw = pd.read_csv(CSV_FILE, sep=";", low_memory=False)

print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())

print("\nFirst 5 rows:")
display(df_raw.head())

✓ CSV file found: \\bca-org-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\New Meet_dataset\NEW MEET_dataset\new meet_raw\new meet_raw\new_meet_raw.csv
File size: 236,243,700 bytes
Shape: (500776, 97)

Columns:
['IN_VALID', 'IN_NUM', 'IN_PSEUDO', 'IN_PASSWORD', 'IN_SEXE', 'IN_AGE', 'IN_DEP', 'IN_VILLE', 'IN_TAILLE', 'IN_POIDS', 'IN_YEUX', 'IN_CHEVEUX', 'IN_ORIGINE', 'IN_ORIGINEC', 'IN_PROF', 'IN_PERS', 'IN_STYLE', 'IN_R1', 'IN_R2', 'IN_R3', 'IN_RECH', 'IN_DESC', 'IN_EMAIL', 'IN_EMAILNV', 'IN_DATECREE', 'IN_TIMECREE', 'IN_DATEPASS', 'IN_TIMEPASS', 'IN_PHOTO', 'IN_PHOTO_BKP', 'IN_MODIF', 'IN_ADMDIAL', 'IN_IP', 'IN_CONN', 'IN_PORT', 'IN_NOIR', 'IN_CGU', 'IN_DATEER', 'IN_PAYS', 'IN_STATUT', 'IN_ENFANTS', 'IN_ENFANTSDES', 'IN_FUMEUR', 'IN_OPTIN', 'IN_A1', 'IN_A2', 'IN_A3', 'IN_A4', 'IN_A5', 'IN_A6', 'IN_A7', 'IN_A8', 'IN_A9', 'IN_A10', 'in_spam', 'in_abond1', 'in_abond2', 'in_visu', 'in_mailo1', 'in_mailo2', 'in_mailo3', 'in_mailo4', 'in_motiv', 'in_att', 'in_netude', 'in_t

,IN_VALID,IN_NUM,IN_PSEUDO,IN_PASSWORD,IN_SEXE,IN_AGE,IN_DEP,IN_VILLE,IN_TAILLE,IN_POIDS,...,in_spokenLang,in_rech_country,in_rech_dep,in_rech_ager1,in_rech_ager2,in_rech_ci_num_km,in_rech_ci_geonames_km,in_measure,in_dateer_last,in_dateer_remove
0,O,2,Phil67,tulipesev,H,23,67,Valff,138.0,44.0,...,"{""6"":""2"",""8"":""4""}",111,NaN,18,27,16386_70,3169070_93,0,2000-01-01,2000-01-01
1,X,443,Nono50,1965,H,47,50,NaN,165.0,60.0,...,NaN,0,NaN,0,0,0,0,0,2000-01-01,2000-01-01
2,O,2186,Jean jacques,azde,H,60,03,Mons,174.0,85.0,...,NaN,0,NaN,0,0,0,0,0,2000-01-01,2000-01-01
3,O,3441747,Bridge1,70807080,H,42,68,NaN,182.0,80.0,...,NaN,0,NaN,0,0,0,0,0,2000-01-01,2000-01-01
4,O,6342,Vincent59,DOMINIQUE,H,42,59,NaN,170.0,65.0,...,NaN,0,NaN,0,0,0,0,0,2000-01-01,2000-01-01


### Post-Cleaning Validation
Validate the cleaned dataset structure, completeness, and selected transformations.

Some datasets contain:

broken rows

extra separators

incomplete records

These can affect row counts. # verying total number of lines in the dataset

In [6]:
with open(CSV_FILE, "r", encoding="utf-8", errors="ignore") as f:
    line_count = sum(1 for line in f)

print("Total lines in file:", line_count)

Total lines in file: 603972


### Verify Record Count
Compare raw file line count with the number of rows loaded into the dataframe.

In [7]:
print("Rows loaded into dataframe:", len(df_raw))

Rows loaded into dataframe: 500776


Read the raw file using a flexible parser and skip malformed rows for inspection.

Confirm the number of rows and columns after cleaning and column selection.

In [8]:
# Ensure df_core is defined before this cell
if 'df_core' not in locals():
    # Define the important indexes for the core dataset
    important_indexes = [
        1, 2, 3, 79, 22, 4, 5, 7, 6, 38, 14, 86
    ]

    # Select the important columns for the core dataset
    df_core = df_raw.iloc[:, important_indexes]

    # Identify and save the garbage columns
    garbage_indexes = [i for i in range(len(df_raw.columns)) if i not in important_indexes]
    df_garbage = df_raw.iloc[:, garbage_indexes]

    # Save the garbage columns to a CSV file
    GARBAGE_FILE = GARBAGE_DIR / "dropped_columns_garbage.csv"
    df_garbage.to_csv(GARBAGE_FILE, index=False)

    print("Garbage file saved to:", GARBAGE_FILE)

# Print the cleaned dataset shape
print("Cleaned dataset shape:", df_core.shape)

Garbage file saved to: \\bca-org-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\New Meet_dataset\NEW MEET_dataset\new meet_raw\new meet_garbage\dropped_columns_garbage.csv
Cleaned dataset shape: (500776, 12)


### Translate Column Names
Rename non-English column headers to English for clarity and consistency.

In [9]:
# Ensure df_raw is defined before this cell
# If not already defined, load the raw CSV file
if 'df_raw' not in locals():
    CSV_FILE = RAW_DIR / "new_meet_raw.csv"
    if CSV_FILE.exists():
        print(f"✓ CSV file found: {CSV_FILE}")
        print(f"File size: {CSV_FILE.stat().st_size:,} bytes")
        df_raw = pd.read_csv(CSV_FILE, sep=";", low_memory=False)
    else:
        raise FileNotFoundError(f"Missing file: {CSV_FILE}")

# Print all column names to verify the column exists
print("Columns in df_raw before renaming:")
print(df_raw.columns.tolist())

# Ensure no leading/trailing spaces in column names
df_raw.columns = df_raw.columns.str.strip()

# Verify the column renaming dictionary
column_translation = {
    "IN_VALID": "is_valid",
    "IN_NUM": "user_id",
    "IN_PSEUDO": "username",
    "IN_PASSWORD": "password",
    "IN_SEXE": "gender",
    "IN_AGE": "age",
    "IN_DEP": "department_code",  # Ensure this matches the original column name in the CSV
    "IN_VILLE": "city",
    "IN_TAILLE": "height",
    "IN_POIDS": "weight",
    "IN_YEUX": "eye_color",
    "IN_CHEVEUX": "hair_color",
    "IN_ORIGINE": "origin",
    "IN_PROF": "profession",
    "IN_STYLE": "style",
    "IN_RECH": "search_preferences",
    "IN_DESC": "description",
    "IN_EMAIL": "email",
    "IN_DATECREE": "account_created_date",
    "IN_TIMECREE": "account_created_time",
    "IN_DATEPASS": "last_password_change_date",
    "IN_TIMEPASS": "last_password_change_time",
    "IN_PHOTO": "photo",
    "IN_IP": "ip_address",
    "IN_CONN": "connection_count",
    "IN_PORT": "port",
    "IN_PAYS": "country",
    "IN_STATUT": "status",
    "IN_ENFANTS": "has_children",
    "IN_ENFANTSDES": "children_description",
    "IN_FUMEUR": "smoker",
    "IN_PRENOM": "first_name",
    "IN_DATEER": "last_error_date",
    "IN_DATEER_LAST": "last_error_timestamp"
}

# Rename columns
df_raw.rename(columns=column_translation, inplace=True)

# Print all column names after renaming
print("Columns in df_raw after renaming:")
print(df_raw.columns.tolist())

# Check if 'department_code' exists
if "department_code" in df_raw.columns:
    department_code_index = df_raw.columns.get_loc("department_code")
    print("Index of 'department_code':", department_code_index)
else:
    print("'department_code' column not found. Check the column names or renaming logic.")

Columns in df_raw before renaming:
['IN_VALID', 'IN_NUM', 'IN_PSEUDO', 'IN_PASSWORD', 'IN_SEXE', 'IN_AGE', 'IN_DEP', 'IN_VILLE', 'IN_TAILLE', 'IN_POIDS', 'IN_YEUX', 'IN_CHEVEUX', 'IN_ORIGINE', 'IN_ORIGINEC', 'IN_PROF', 'IN_PERS', 'IN_STYLE', 'IN_R1', 'IN_R2', 'IN_R3', 'IN_RECH', 'IN_DESC', 'IN_EMAIL', 'IN_EMAILNV', 'IN_DATECREE', 'IN_TIMECREE', 'IN_DATEPASS', 'IN_TIMEPASS', 'IN_PHOTO', 'IN_PHOTO_BKP', 'IN_MODIF', 'IN_ADMDIAL', 'IN_IP', 'IN_CONN', 'IN_PORT', 'IN_NOIR', 'IN_CGU', 'IN_DATEER', 'IN_PAYS', 'IN_STATUT', 'IN_ENFANTS', 'IN_ENFANTSDES', 'IN_FUMEUR', 'IN_OPTIN', 'IN_A1', 'IN_A2', 'IN_A3', 'IN_A4', 'IN_A5', 'IN_A6', 'IN_A7', 'IN_A8', 'IN_A9', 'IN_A10', 'in_spam', 'in_abond1', 'in_abond2', 'in_visu', 'in_mailo1', 'in_mailo2', 'in_mailo3', 'in_mailo4', 'in_motiv', 'in_att', 'in_netude', 'in_tcarac', 'in_ager1', 'in_ager2', 'in_datenais', 'in_facebookid', 'in_facebook_secret', 'in_chatonline', 'in_country', 'in_preflanguage', 'in_mailcounterlevel', 'in_newmailsincelastlogon', 'in_fa

### Display All Columns
Show every column name with its index position for easier reference.

In [10]:
column_index_table = pd.DataFrame({
    "Index": range(len(df_raw.columns)),
    "Column_Name": df_raw.columns
})

display(column_index_table)

,Index,Column_Name
0,0,is_valid
1,1,user_id
2,2,username
3,3,password
4,4,gender
...,...,...
92,92,in_rech_ci_num_km
93,93,in_rech_ci_geonames_km
94,94,in_measure
95,95,in_dateer_last


In [ ]:
# Ensure df_raw is defined before using it
if 'df_raw' not in locals():
    CSV_FILE = RAW_DIR / "new_meet_raw.csv"
    if CSV_FILE.exists():
        df_raw = pd.read_csv(CSV_FILE, sep=";", low_memory=False)
    else:
        raise FileNotFoundError(f"Missing file: {CSV_FILE}")

print("Columns in df_raw:")
print(df_raw.columns.tolist())

# Ensure no leading/trailing spaces in column names
df_raw.columns = df_raw.columns.str.strip()

Columns in df_raw:
['is_valid', 'user_id', 'username', 'password', 'gender', 'age', 'department_code', 'city', 'height', 'weight', 'eye_color', 'hair_color', 'origin', 'IN_ORIGINEC', 'profession', 'IN_PERS', 'style', 'IN_R1', 'IN_R2', 'IN_R3', 'search_preferences', 'description', 'email', 'IN_EMAILNV', 'account_created_date', 'account_created_time', 'last_password_change_date', 'last_password_change_time', 'photo', 'IN_PHOTO_BKP', 'IN_MODIF', 'IN_ADMDIAL', 'ip_address', 'connection_count', 'port', 'IN_NOIR', 'IN_CGU', 'last_error_date', 'country', 'status', 'has_children', 'children_description', 'smoker', 'IN_OPTIN', 'IN_A1', 'IN_A2', 'IN_A3', 'IN_A4', 'IN_A5', 'IN_A6', 'IN_A7', 'IN_A8', 'IN_A9', 'IN_A10', 'in_spam', 'in_abond1', 'in_abond2', 'in_visu', 'in_mailo1', 'in_mailo2', 'in_mailo3', 'in_mailo4', 'in_motiv', 'in_att', 'in_netude', 'in_tcarac', 'in_ager1', 'in_ager2', 'in_datenais', 'in_facebookid', 'in_facebook_secret', 'in_chatonline', 'in_country', 'in_preflanguage', 'in_mai

### Core Columns Selection
Keep essential user identity, demographic, and location fields for analysis.

In [11]:
important_indexes = [
    1, 2, 3, 79, 22, 4, 5, 7, 6, 38, 14, 86
]

df_core = df_raw.iloc[:, important_indexes]

garbage_indexes = [i for i in range(len(df_raw.columns)) if i not in important_indexes]

df_garbage = df_raw.iloc[:, garbage_indexes]

GARBAGE_FILE = GARBAGE_DIR / "dropped_columns_garbage.csv"

df_garbage.to_csv(GARBAGE_FILE, index=False)

print("Garbage file saved to:", GARBAGE_FILE)

Garbage file saved to: \\bca-org-00\Users Folder\jalleyne\Desktop\Protexxa\data_cleaning_scripts\New Meet_dataset\NEW MEET_dataset\new meet_raw\new meet_garbage\dropped_columns_garbage.csv


### Preview Core Dataset
Display the first few rows of the cleaned dataset containing the selected columns.

In [12]:
df_core.head()

,user_id,username,password,in_lastname,email,gender,age,city,department_code,country,profession,first_name
0,2,Phil67,tulipesev,NaN,oxygenix@netcourrier.com,H,23,Valff,67,1,8,phillipe
1,443,Nono50,1965,NaN,!chesnaye50@hotmail.com,H,47,NaN,50,1,NaN,NaN
2,2186,Jean jacques,azde,NaN,jj_delannoy@hotmail.com,H,60,Mons,03,2,NaN,NaN
3,3441747,Bridge1,70807080,NaN,byly1970@hotmail.com,H,42,NaN,68,1,Finance,NaN
4,6342,Vincent59,DOMINIQUE,NaN,jvg1@libertysurf.fr,H,42,NaN,59,1,Enseignant,NaN


### Create Full Name
Merge first_name and in_lastname into a single full_name column.

In [13]:
df_core["full_name"] = df_core["first_name"] + " " + df_core["in_lastname"]

df_core.drop(columns=["first_name", "in_lastname"], inplace=True)

C:\Users\jalleyne\AppData\Local\Temp\ipykernel_24936\985000714.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_core["full_name"] = df_core["first_name"] + " " + df_core["in_lastname"]
C:\Users\jalleyne\AppData\Local\Temp\ipykernel_24936\985000714.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_core.drop(columns=["first_name", "in_lastname"], inplace=True)


### Missing Values Check
Review null values in each cleaned column after transformation.

In [14]:
print("Missing values per column:")
print(df_core.isnull().sum())

Missing values per column:
user_id                 0
username                1
password                3
email                  12
gender                590
age                     0
city               142125
department_code    363244
country                 0
profession         171500
full_name          498027
dtype: int64


### Duplicate Check
Identify duplicate records remaining in the cleaned dataset.

In [15]:
print("Duplicate rows:", df_core.duplicated().sum())

Duplicate rows: 0


### Data Type Validation
Confirm each cleaned column has an appropriate data type.

In [16]:
print(df_core.dtypes)

user_id             int64
username           object
password           object
email              object
gender             object
age                 int64
city               object
department_code    object
country             int64
profession         object
full_name          object
dtype: object
